In [9]:
from pathlib import Path

DATA_DIR = Path("/content/ITS-012012203802/data/processed")

for split in ["train", "val", "test"]:

    image_dir = DATA_DIR / "images" / split
    label_dir = DATA_DIR / "labels" / split

    images = list(image_dir.glob("*")) if image_dir.exists() else []
    labels = list(label_dir.glob("*.txt")) if label_dir.exists() else []

    print(f"\n{split.upper()}")
    print("Số ảnh:", len(images))
    print("Số nhãn:", len(labels))

    if len(images) == 0:
        raise ValueError(f"Không có ảnh trong {image_dir}")

    if len(labels) == 0:
        raise ValueError(f"Không có file nhãn trong {label_dir}")


TRAIN
Số ảnh: 3
Số nhãn: 3

VAL
Số ảnh: 1
Số nhãn: 1

TEST
Số ảnh: 0
Số nhãn: 0


ValueError: Không có ảnh trong /content/ITS-012012203802/data/processed/images/test

In [10]:
import yaml
from pathlib import Path

ROOT = Path("/content/ITS-012012203802")

# Đọc train_config.yaml
with open(ROOT / "configs/train_config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Đọc dataset.yaml
with open(ROOT / "configs/dataset.yaml", "r") as f:
    dataset = yaml.safe_load(f)

# Chỉnh đường dẫn dataset cho môi trường Colab
dataset["path"] = str(DATA_DIR)

# Tạo file cấu hình dataset dành cho Colab
colab_dataset = "/content/dataset_colab.yaml"

with open(colab_dataset, "w") as f:
    yaml.safe_dump(dataset, f, sort_keys=False)

print("Đã đọc cấu hình thành công!")
print("Dataset:", colab_dataset)
print("Epochs:", config["training"]["epochs"])
print("Batch size:", config["training"]["batch_size"])

Đã đọc cấu hình thành công!
Dataset: /content/dataset_colab.yaml
Epochs: 50
Batch size: 16


In [11]:
from ultralytics import YOLO

# Khởi tạo mô hình YOLOv8 Nano
model = YOLO(config["model_type"])

# Lấy thông số huấn luyện
training = config["training"]

# Bắt đầu huấn luyện
results = model.train(
    data=colab_dataset,

    epochs=training["epochs"],
    batch=training["batch_size"],
    imgsz=training["image_size"],

    lr0=training["learning_rate"],
    optimizer=training["optimizer"],

    device=0,

    project="/content/ITS-012012203802/outputs",
    name="traffic_sign",

    workers=2,
    patience=20,
    plots=True,
    save=True,
    exist_ok=True
)

print("HUẤN LUYỆN HOÀN TẤT!")
print("Kết quả:", results.save_dir)

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=traffic_sign, nbs=64, nms=None, opset=None, optimize=False, o

In [12]:
from pathlib import Path

run_dir = Path(results.save_dir)

best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"

print("Best model:", best_pt.exists())
print("Last model:", last_pt.exists())

print("Đường dẫn best.pt:", best_pt)

Best model: True
Last model: True
Đường dẫn best.pt: /content/ITS-012012203802/outputs/traffic_sign/weights/best.pt


In [13]:
from ultralytics import YOLO

# Load trọng số tốt nhất
best_model = YOLO(str(best_pt))

# Xuất ONNX
onnx_path = best_model.export(
    format="onnx",
    imgsz=640
)

print("Đã xuất ONNX:", onnx_path)

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 73 layers, 3,133,436 parameters, 0 gradients, 8.6 GFLOPs

PyTorch: starting from '/content/ITS-012012203802/outputs/traffic_sign/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (6.2 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 12 packages in 214ms
Prepared 3 packages in 326ms
Installed 3 packages in 9ms
 + colorama==0.4.6
 + onnxruntime==1.31.1
 + onnxslim==0.1.99

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxslim 0.1.99...
ON

In [14]:
import shutil
from pathlib import Path

# Thư mục bàn giao trên Google Drive
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/ITS-012012203802"
)

checkpoint_dir = DRIVE_ROOT / "outputs/checkpoints"
log_dir = DRIVE_ROOT / "outputs/logs"

checkpoint_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

# Sao chép best.pt và last.pt
shutil.copy2(
    best_pt,
    checkpoint_dir / "best.pt"
)

shutil.copy2(
    last_pt,
    checkpoint_dir / "last.pt"
)

# Sao chép best.onnx
onnx_file = Path(onnx_path)

shutil.copy2(
    onnx_file,
    checkpoint_dir / "best.onnx"
)

# Sao chép toàn bộ kết quả huấn luyện
shutil.copytree(
    run_dir,
    log_dir / "traffic_sign",
    dirs_exist_ok=True
)

print("ĐÃ LƯU KẾT QUẢ VÀO GOOGLE DRIVE!")
print("Checkpoint:", checkpoint_dir)
print("Logs:", log_dir)

ĐÃ LƯU KẾT QUẢ VÀO GOOGLE DRIVE!
Checkpoint: /content/drive/MyDrive/ITS-012012203802/outputs/checkpoints
Logs: /content/drive/MyDrive/ITS-012012203802/outputs/logs
